
# DAPPC - LAB 2
## Identification of Homogeneous Groups of Patients with SOMs

In this notebook you will:

1. Load a prepared dataset  
2. Select the variables used for clustering  
3. Normalize the input features  
4. Define and train a square Self-Organizing Map (SOM)  
5. Extract the weight matrix of the trained SOM  
6. Apply hierarchical clustering on neuron weights  
7. Visualize the dendrogram and define the number of clusters  
8. Build the sample hits map and identify contiguous neuron bubbles  
9. Assign each subject to a winner neuron, a neuron cluster, and a bubble  
10. Save the results

We assume:
- The dataset is already prepared and filled (it has no MVs)
- Each row corresponds to one subject
- Subject identifiers are available in the first six columns of the dataset


## 0) Setup


In [282]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from matplotlib.patches import RegularPolygon
from matplotlib import cm, colors

from collections import deque
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from minisom import MiniSom


## 1) Load Dataset


In [ ]:
# === TODO: update path and sheet name ===
file_path = ...
sheet_name = ...

df = pd.read_excel(...)

print("Initial shape:", df.shape)
df.head()


## 2) Column Index Table


In [ ]:
display(pd.DataFrame({
    "index": np.arange(df.shape[1]),
    "column_name": df.columns,
    "dtype": df.dtypes
}))



## 3) Select Variables for SOM

Define here the variables that will be used to build the SOM.

You can specify them either:
- by column indices, or  
- by explicit column names.

Remember that we're going to use a clustering method (unsupervised learning approach).

In [ ]:

# ============================================================
# 1. Define variable groups: you should obtain the names of the all the input features 
# ============================================================
input_features = ...
X = df[input_features]
print("Number of selected variables:", len(input_features))
print("First selected variable:", input_features[0])
print("Last selected variable:", input_features[-1])



## 4) Normalize the Input Features

Since SOM training is distance-based, the variables must be on a comparable scale. Here we use min-max scaling.


In [ ]:
scaler = ...
X_norm = ...


## 6) Define and Train the SOM

Main steps:
1. Define the dimension of the square SOM  
2. Create the SOM architecture  
3. Initialize the neuron weights  
4. Train the SOM


In [ ]:
# ============================================================
# 1. Define the dimensions of the SOM (square SOM)
# ============================================================
n = ...

# ============================================================
# 2. Create the SOM architecture
# ============================================================
som = MiniSom(
    x=n,
    y=n,
    input_len=X_norm.shape[1],
    sigma=4,
    learning_rate=0.3,
    activation_distance='euclidean',
    topology='hexagonal',
    neighborhood_function='gaussian',
)

# ============================================================
# 3. Initialize and train the SOM
# ============================================================
som.random_weights_init(X_norm)
som.train(X_norm, num_iteration=200, use_epochs=True) #it may take a while (usually around 5-10 minutes)

print(f"Trained SOM: {n} x {n}")


## 7) Save the Trained SOM

The trained SOM can be saved and reused later without retraining.


In [ ]:
 # adjust the filename and the directory in which you want to save the SOMs
som_filename = f"SOM_Dim_{n}x{n}.pkl"

with open(som_filename, 'wb') as f:
    pickle.dump(som, f)

print("Saved SOM to:", som_filename)



## 8) Extract the Weight Matrix

Each neuron of the trained SOM is associated with one weight vector.

We extract the neuron weights and reshape them into a 2D matrix:
- one row per neuron  
- one column per feature


In [ ]:
# Check that the wight shapes are (number of neurons, number of features)
weights_3d = som.get_weights()         
weights = weights_3d.reshape(n * n, X.shape[1])
print("Weight matrix shape:", weights.shape)



## 9) Hierarchical Clustering on SOM Weights

We now apply agglomerative hierarchical clustering to the neuron weights.

As in the previous laboratory, we use:
- complete linkage  
- cityblock distance

Yuo're appling hierarchical clustering on SOM's neurons so the numbers on the x-axis represent the position of each neuron in the 2D Network -> the first (neuron number 0) is the top-left corner, the last one is the right-bottom corner.

In [311]:
# ============================================================
# Compute the hierarchical clustering tree
# ============================================================
Z = linkage(weights, method='complete', metric='cityblock')

# ============================================================
# Visualize the dendrogram and try to identify the nubmer of clusters of neurons
# ============================================================
fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(Z, color_threshold=0) # adjust threshold according to the number of cluster you want to obtain
ax.tick_params(axis='x', labelsize=10, labelrotation=90)   
ax.set_title('Dendrogram of SOM Neuron Weights')
ax.set_xlabel('Neurons')
ax.set_ylabel('Distance')
plt.show()



## 10) Define the Number of Clusters

Choose the number of clusters by cutting the dendrogram.

The variable `numCl` can be changed according to the desired partition.


In [ ]:
# you may need to insert some codes above to define the number of cluster (compute the metrics and evaluate them)
numCl = ... # define the number of cluster based on the dendrogram and the metrics you've computed

# Cluster label of each neuron
clusterIdx = ... 
# 2D cluster map of the SOM neurons
cluster_map = clusterIdx.reshape(n, n)
print("Number of neuron clusters:", numCl) 



## 12) Identify Contiguous Neuron Bubbles

A bubble is defined as a group of neighboring neurons that:
- belong to the same neuron cluster  
- are contiguous on the SOM grid

Here we identify the connected components of the 2D cluster map using 4-neighborhood connectivity.
The function below helps you to visualize the bubble of neurons (i.e., groups of neighbour neurons). You can use this function to check if the groups are contigous or not.

The function create a figure with two subplots:
- On the lift side you can see the map of victories of the SOM. The number inside each neuron is the number of victories.
- On the righ side, you can see the neurons divided into N groups (N is the umber of clusters you defined before) with a colormap. The number inside is the cluster label.

You can also modify the function if you want to plot other information rather than the default ones.

In [313]:
def plot_som_hits_and_bubbles_side_by_side(
    som,
    hits,
    cluster_map,
    hex_radius=0.85 / np.sqrt(3),
    hits_cmap='Blues',
    bubbles_cmap='jet',
    hits_title='Hits Map',
    bubbles_title='Neuron Bubbles'
):
    """
    Plot the SOM hits map and the neuron bubble map side by side.

    Parameters
    ----------
    som : MiniSom
        Trained SOM object.
    hits : ndarray of shape (n_rows, n_cols)
        Number of wins for each neuron.
    cluster_map : ndarray of shape (n_rows, n_cols)
        Cluster label assigned to each neuron.
    hex_radius : float, default=0.85 / sqrt(3)
        Radius of each hexagon.
    hits_cmap : str, default='Blues'
        Colormap used for the hits map.
    bubbles_cmap : str, default='jet'
        Colormap used for the bubble map.
    hits_title : str, default='Hits Map'
        Title of the hits subplot.
    bubbles_title : str, default='Neuron Bubbles'
        Title of the bubble subplot.
    """

    # --------------------------------------------------------
    # 1. Extract SOM geometry
    # --------------------------------------------------------
    weights_3d = som.get_weights()
    n_rows, n_cols, _ = weights_3d.shape
    xx, yy = som.get_euclidean_coordinates()

    # --------------------------------------------------------
    # 2. Colormaps
    # --------------------------------------------------------
    hits_cmap_obj = cm.get_cmap(hits_cmap)
    max_hits = np.max(hits) if np.max(hits) > 0 else 1
    hits_norm = colors.Normalize(vmin=0, vmax=max_hits)

    num_clusters = int(np.max(cluster_map))
    bubbles_cmap_obj = cm.get_cmap(bubbles_cmap, num_clusters)

    # --------------------------------------------------------
    # 3. Create figure and axes
    # --------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    ax_hits, ax_bubbles = axes

    # --------------------------------------------------------
    # 4. Common axis formatting
    # --------------------------------------------------------
    def format_axis(ax):
        ax.set_aspect('equal')

        # Tick labels arranged for didactic reading:
        # left-to-right, top-to-bottom
        ax.set_xticks(np.arange(n_cols))
        ax.set_yticks(np.arange(n_rows) * np.sqrt(3) / 2)

        ax.set_xticklabels(np.arange(n_cols))
        ax.set_yticklabels(np.arange(n_rows - 1, -1, -1))

        yy_plot = yy * np.sqrt(3) / 2
        ax.set_xlim(xx.min() - 1, xx.max() + 1)
        ax.set_ylim(yy_plot.min() - 1, yy_plot.max() + 1)

    # --------------------------------------------------------
    # 5. Left subplot: hits map
    # --------------------------------------------------------
    for i in range(n_rows):
        for j in range(n_cols):
            # Visual arrangement:
            # row i is displayed from top to bottom
            # column j is displayed from left to right
            i_plot = n_rows - 1 - i
            j_plot = j

            wx = xx[(j_plot, i_plot)]
            wy = yy[(j_plot, i_plot)] * np.sqrt(3) / 2

            hit_value = hits[i, j]
            face_color = hits_cmap_obj(hits_norm(hit_value))

            hexagon = RegularPolygon(
                (wx, wy),
                numVertices=6,
                radius=hex_radius,
                facecolor=face_color,
                edgecolor='lightgray',
                linewidth=1.2
            )
            ax_hits.add_patch(hexagon)

            text_color = 'white' if hit_value > 0.6 * max_hits else 'black'
            ax_hits.text(
                wx, wy,
                str(hit_value),
                ha='center',
                va='center',
                fontsize=10,
                fontweight='bold',
                color=text_color
            )

    format_axis(ax_hits)
    ax_hits.set_title(hits_title, fontweight='bold')

    # --------------------------------------------------------
    # 6. Right subplot: bubble map
    # --------------------------------------------------------
    for i in range(n_rows):
        for j in range(n_cols):
            i_plot = n_rows - 1 - i
            j_plot = j

            wx = xx[(j_plot, i_plot)]
            wy = yy[(j_plot, i_plot)] * np.sqrt(3) / 2

            cluster_id = cluster_map[i, j]
            cluster_color = bubbles_cmap_obj(cluster_id - 1)

            hexagon = RegularPolygon(
                (wx, wy),
                numVertices=6,
                radius=hex_radius,
                facecolor=cluster_color,
                edgecolor=cluster_color,
                alpha=0.28,
                linewidth=2
            )
            ax_bubbles.add_patch(hexagon)

            ax_bubbles.text(
                wx, wy,
                str(cluster_id),
                ha='center',
                va='center',
                fontsize=10,
                fontweight='bold',
                color='black'
            )

    format_axis(ax_bubbles)
    ax_bubbles.set_title(bubbles_title, fontweight='bold')

    plt.tight_layout()
    plt.show()

Here there's an example of how to use the function as it's written right now. If you change something in the function you may have to adjust also how you call it.

In [ ]:
# Compute som hits (wins) 
hits = np.zeros((n, n), dtype=int)
for x in X_norm:
    i, j = som.winner(x)
    hits[i, j] += 1

# Plot som hits and bubbles
plot_som_hits_and_bubbles_side_by_side(
    som=som,
    hits=hits,
    cluster_map=cluster_map
)


## 14) Assign Subjects to Neurons, Clusters, and Bubbles

Each subject is assigned to:
- the winner neuron  
- the cluster of the winner neuron  
- the contiguous bubble containing the winner neuron


In [ ]:
# ============================================================
# Assign each subject to the cluster of its winner neuron
# ============================================================
# Example: subject IDs
# Replace this with your real subject ID variable
# subject_ids = df['subject_id'].values

subject_winners = []
subject_clusters = []

for x in X_norm:
    # Winner neuron of the current subject
    winner = ...

    # Cluster (bubble) of the winner neuron
    cluster_id = ...

    subject_winners.append(winner)
    subject_clusters.append(cluster_id)

subject_winners = np.array(subject_winners)
subject_clusters = np.array(subject_clusters)

# Build a summary table
subjects_clustered = pd.DataFrame({
    'subject_id': df['subject_id'].values,
    'cluster_id': subject_clusters
})

subjects_clustered.head()


## 15) Explore the Subjects in Each Cluster

The following cell prints the number of subjects assigned to each cluster and each bubble. Modify it if you want to analyze something more.


In [ ]:

print("Subjects per cluster:")
cluster_summary = (
    subjects_clustered
    .groupby('cluster_id')
    .size()
    .reset_index(name='n_subjects')
    .sort_values(by='cluster_id')
)

cluster_summary


## 16) Save the Final Results

Save the subject assignments as a file (.csv or .xlsx or whatever you prefer).

You can also save one file per cluster if you prefer. Remember to restore the information about the IDs (first 6 columns) and the outcome (last column) of the initial dataset.


In [ ]:

results_filename = ...
print("Saved subject assignments to:", results_filename)
